# HGT and pair-conditioned gated HGT on PrimeKG — Colab T4 (Member 2)

**Before running:** Runtime → Change runtime type → **T4 GPU**, and upload `pcgate_colab_bundle.zip`
(built with `python scripts/make_colab_bundle.py`) to the top level of *My Drive*.

Everything is written to `My Drive/pcgate/results/primekg_eligible/`, next to Member 1's GraphSAGE and
popularity results, so `summarize` produces the paired comparisons for every model. Every step is
resumable: after a disconnect re-run cells 1–4, then continue where you stopped.

| section | what it does | T4 time (rough) |
|---|---|---|
| 1–4 | GPU, Drive, PyG, data + tests | 2–15 min |
| 5 | memory / runtime profile of hgt and gated | 3 min |
| 6 | **bounded tuning** on validation data → frozen config | 25–50 min |
| 7 | **main experiments**: hgt + gated × 3 settings × 3 seeds | 1–2 h |
| 8 | **trained controls**: uniform + param_matched on disease-held-out | 20–40 min |
| 9 | summaries, paired differences, coverage | 2 min |
| 10 | **gate analysis** + evaluation-time gate controls | 5 min |
| 11 | explanation case study + candidate table (gated) | 10–20 min |
| 12 | download results and the frozen configs | 1 min |

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Drive and unpack the bundle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/pcgate && mkdir -p /content/pcgate
!unzip -q /content/drive/MyDrive/pcgate_colab_bundle.zip -d /content/pcgate
%cd /content/pcgate
!git log --oneline -1 2>/dev/null; cat BUNDLE_INFO.txt 2>/dev/null; ls

## 3. Install PyTorch Geometric (Colab already has torch, pandas, scikit-learn, psutil)

In [ ]:
!pip install -q torch_geometric pyyaml
import torch, torch_geometric
print(torch.__version__, torch_geometric.__version__, 'cuda:', torch.cuda.is_available())

## 4. Data, tests, and Member 1's committed results

If the bundle already contains `data/processed/primekg` (ask Member 1, or build it locally) this only
verifies it. Otherwise it downloads PrimeKG (~1 GB) and runs `prepare`, which rebuilds the same splits
from the same seed and checks their hashes.

In [ ]:
import os
if not os.path.exists('data/processed/primekg/splits/disease_s0_with_indication/manifest.json'):
    !python -m pcgate download --config configs/primekg_colab_eligible.yaml
    !python -m pcgate prepare  --config configs/primekg_colab_eligible.yaml
!python -m pytest -q tests -p no:cacheprovider
!python -c "from pcgate.utils import read_json; [print(s, read_json(f'data/processed/primekg/splits/{s}_s0_with_indication/integrity_checks.json')['all_passed']) for s in ('random','drug','disease')]"
# Member 1's sage/popularity metrics travel in the repo; copy without overwriting anything on Drive
!mkdir -p /content/drive/MyDrive/pcgate/results/primekg_eligible && cp -rn results/primekg_eligible/* /content/drive/MyDrive/pcgate/results/primekg_eligible/ 2>/dev/null; echo copied

## 5. Memory / runtime profile (W3) for the HGT family

In [ ]:
!python -m pcgate profile --config configs/primekg_colab_eligible.yaml --models hgt gated

## 6. Bounded tuning — validation data only (milestone: Nov 18)

Same 4 configurations for `hgt` and `gated`, on the disease-held-out validation pool, seed 0. The test
pool is never scored. `uniform` / `param_matched` inherit the gated values automatically.

Trials are saved to `tuning/trials.csv` on Drive after each run, so if the runtime disconnects,
re-running this cell skips finished trials and still rewrites the frozen config at the end.

**Commit the printed frozen config** (`configs/primekg_eligible_frozen.yaml`, downloaded in section 12)
— that is the "configurations frozen" checkpoint. Do not re-tune after looking at test results.

In [ ]:
!python -m pcgate tune --config configs/primekg_colab_eligible_tune.yaml --only hgt gated --freeze-to configs/primekg_colab_eligible_frozen.yaml
# local twin of the frozen config (same overrides, repo paths) for committing to GitHub
import yaml, shutil
fz = yaml.safe_load(open('configs/primekg_colab_eligible_frozen.yaml'))
with open('configs/primekg_eligible_frozen.yaml', 'w') as f:
    f.write('# FROZEN configuration - written by `pcgate tune` (Colab) from validation data only.\n')
    yaml.safe_dump({'base': 'primekg_eligible_tune.yaml', 'model_overrides': fz['model_overrides']}, f, sort_keys=False)
for p in ('configs/primekg_colab_eligible_frozen.yaml', 'configs/primekg_eligible_frozen.yaml'):
    shutil.copy(p, '/content/drive/MyDrive/pcgate/results/primekg_eligible/tuning/')
print(open('configs/primekg_eligible_frozen.yaml').read())

### If the runtime restarted after tuning

`/content` is wiped on restart; the frozen configs are also on Drive. Run this before sections 7–11.

In [ ]:
import os, shutil
src = '/content/drive/MyDrive/pcgate/results/primekg_eligible/tuning/'
for f in ('primekg_colab_eligible_frozen.yaml', 'primekg_eligible_frozen.yaml'):
    if os.path.exists(src + f): shutil.copy(src + f, 'configs/' + f); print('restored', f)

## 7. Main experiments — HGT and gated HGT, all three settings, 3 seeds (milestone: Nov 23)

Checkpoints (`model.pt`) are saved next to each run's `metrics.json` on Drive.

In [ ]:
!python -m pcgate run --config configs/primekg_colab_eligible_frozen.yaml --only hgt gated

## 8. Trained gate controls on disease-held-out (milestone: Nov 26)

`uniform`: same relation summaries, equal weights. `param_matched`: same summaries, non-gated decoder
with the gated decoder's parameter count. Both use the gated model's frozen hyperparameters.

In [ ]:
!python -m pcgate run --config configs/primekg_colab_eligible_frozen.yaml --only uniform param_matched

## 9. Summaries, paired differences, bootstrap CIs, coverage by degree

In [ ]:
!python -m pcgate summarize --config configs/primekg_colab_eligible_frozen.yaml
!python -m pcgate coverage  --config configs/primekg_colab_eligible_frozen.yaml

## 10. Gate analysis — is the gain from pair-conditioned weighting? (RQ3)

Evaluation-time controls on the **same** gated checkpoints: `shuffled_query` (query from an unrelated
training pair), `mean_query` (one constant query → node-only weighting), `uniform_weights`.
Also reports relation weights, entropy, and how much a drug's weights change with the disease.

In [ ]:
!python -m pcgate gates --config configs/primekg_colab_eligible_frozen.yaml --setting disease
!python -m pcgate gates --config configs/primekg_colab_eligible_frozen.yaml --setting random

## 11. Limited explanation case study (E5) and candidate table

Path retrieval + perturbation against matched random removals, for ~10 pairs. Paths show what the
model relies on, not mechanism. Candidates are computational hypotheses, not validated treatments.

In [ ]:
!python -m pcgate explain    --config configs/primekg_colab_eligible_frozen.yaml --model gated
!python -m pcgate candidates --config configs/primekg_colab_eligible_frozen.yaml --model gated | head -20

## 12. Download results (no checkpoints) and the frozen configs

In [ ]:
!cp configs/primekg_eligible_frozen.yaml configs/primekg_colab_eligible_frozen.yaml /content/drive/MyDrive/pcgate/results/primekg_eligible/
!cd /content/drive/MyDrive/pcgate/results/primekg_eligible && zip -qr /content/member2_results.zip . -x '*.pt'
from google.colab import files
files.download('/content/member2_results.zip')